# SUE optimizer test notebook

This notebook compares the current `trust-constr` setup against one or more `SLSQP` settings for the road SUE logic.

It is meant to answer:
- does the optimizer stop too early?
- are OD equality constraints still satisfied?
- do total travel times and route flows look plausible?

Important: `SLSQP` with `ftol=1e5` and `eps=1e5` is intentionally kept here as a test case, but those values are so loose that very early stopping is expected.

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import Bounds, minimize

REPO_ROOT = Path('/Users/laura/Desktop/infraScan_lkuehner/infraScan')
DATA_ROOT = Path('/Volumes/WD_Windows/MSc_Thesis')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from infraScanRoad.scoring import (
    Commonality,
    IntCostFun,
    convert_data_to_input,
    get_nw_data,
)

POINTS_PATH = DATA_ROOT / 'data/infraScanRoad/Network/processed/points_with_attribute.gpkg'
EDGES_PATH = DATA_ROOT / 'data/infraScanRoad/Network/processed/edges_with_attribute.gpkg'
VORONOI_PATH = DATA_ROOT / 'data/infraScanRoad/Voronoi/voronoi_status_quo_euclidian.gpkg'
OD_PATH = DATA_ROOT / 'data/infraScanRoad/traffic_flow/od/od_matrix_20.csv'

THETA = 1.2
BETA_COMMONALITY = 1.0

OPTIMIZER_CONFIGS = {
    'trust_constr_current': {
        'method': 'trust-constr',
        'options': {'maxiter': 3, 'verbose': 0, 'disp': True},
    },
    'slsqp_loose_test': {
        'method': 'SLSQP',
        'options': {'ftol': 1e5, 'eps': 1e5, 'maxiter': 3, 'disp': True},
    },
    'slsqp_tighter_reference': {
        'method': 'SLSQP',
        'options': {'ftol': 1e-6, 'eps': 1e-6, 'maxiter': 200, 'disp': True},
    },
}


In [ ]:
def prepare_sue_inputs(points_path=POINTS_PATH, edges_path=EDGES_PATH, voronoi_path=VORONOI_PATH, od_path=OD_PATH):
    points = gpd.read_file(points_path)
    edges = gpd.read_file(edges_path)
    voronoi = gpd.read_file(voronoi_path)
    od_matrix = pd.read_csv(od_path, sep=',', index_col=0)

    _, _, _, _, _, par = convert_data_to_input(points=points, edges=edges)
    delta_ir, delta_odr, _, D_od, nOD, nroutes, od_pairs = get_nw_data(
        OD_matrix=od_matrix,
        points=points,
        voronoi_gdf=voronoi,
        edges=edges,
    )

    od_single = int(np.sqrt(nOD))
    idx = np.arange(0, nOD, od_single + 1)
    D_od = np.delete(D_od, idx)
    delta_odr = np.delete(delta_odr, idx, axis=0)
    od_pairs = [pair for i, pair in enumerate(od_pairs) if i not in set(idx)]

    fftt_r = np.matmul(par['fftt_i'].transpose(), delta_ir).transpose()
    cf_r = Commonality(BETA_COMMONALITY, delta_ir, delta_odr, par['fftt_i'], fftt_r)

    return {
        'points': points,
        'edges': edges,
        'voronoi': voronoi,
        'od_matrix': od_matrix,
        'delta_ir': delta_ir,
        'delta_odr': delta_odr,
        'D_od': D_od,
        'par': par,
        'cf_r': cf_r,
        'theta': THETA,
        'nroutes': nroutes,
        'od_pairs': od_pairs,
    }


sue_inputs = prepare_sue_inputs()
print('Prepared SUE inputs:')
print('  routes:', sue_inputs['delta_ir'].shape[1])
print('  links:', sue_inputs['delta_ir'].shape[0])
print('  OD pairs after intrazonal removal:', sue_inputs['delta_odr'].shape[0])
print('  total OD demand:', float(np.sum(sue_inputs['D_od'])))


In [ ]:
def build_initial_route_demand(D_od, delta_odr):
    row_sums = np.sum(delta_odr, axis=1).astype(float)
    tt = np.divide(
        D_od.transpose(),
        row_sums,
        out=np.zeros_like(D_od, dtype=float).transpose(),
        where=row_sums != 0,
    ).transpose()
    tt = np.nan_to_num(tt, nan=0.0, posinf=0.0, neginf=0.0)
    D_r0 = np.matmul(delta_odr.transpose(), tt)
    D_r0 = np.nan_to_num(D_r0, nan=0.01, posinf=0.01, neginf=0.01)
    D_r0[D_r0 <= 0] = 0.01
    return D_r0


def run_sue_optimizer(method, options, sue_inputs):
    delta_ir = sue_inputs['delta_ir']
    delta_odr = sue_inputs['delta_odr']
    D_od = sue_inputs['D_od']
    par = sue_inputs['par']
    cf_r = sue_inputs['cf_r']

    def int_links_times(D_r):
        x_i = np.matmul(delta_ir, D_r)
        return IntCostFun(x_i, par)

    def objective(x):
        x_safe = np.array(x, dtype=float, copy=True)
        x_safe[x_safe <= 0] = 0.0001
        temp_log = np.log(x_safe)
        temp_log[np.isinf(temp_log)] = 0.1
        temp_log[np.isnan(temp_log)] = 0.1
        thetavec = np.ones_like(x_safe)
        return float(
            np.sum(int_links_times(x_safe))
            + np.sum(np.divide(np.multiply(x_safe, temp_log), thetavec))
            + np.sum(np.multiply(x_safe, cf_r))
        )

    def eq_residual(x):
        return (np.matmul(delta_odr, x) - D_od).flatten()

    def ineq_residual(x):
        return x

    D_r0 = build_initial_route_demand(D_od, delta_odr)
    lb = np.zeros(np.shape(D_r0)).flatten() + 0.01
    ub = (max(D_od) * np.ones(np.shape(D_r0))).flatten() * 5
    bounds = Bounds(lb, ub)

    res = minimize(
        objective,
        D_r0.flatten(),
        method=method,
        constraints=[
            {'type': 'eq', 'fun': eq_residual},
            {'type': 'ineq', 'fun': ineq_residual},
        ],
        options=options,
        bounds=bounds,
    )

    D_r = np.array(res.x, dtype=float, copy=True)
    D_r[D_r <= 0] = 0.001
    Xi = delta_ir * D_r
    intTrec_i = IntCostFun(Xi, par)
    Xi_array = np.asarray(Xi)
    if Xi_array.ndim > 1:
        link_flows = Xi_array.sum(axis=1).reshape(-1, 1)
    else:
        link_flows = Xi_array.reshape(-1, 1)
    total_travel_time = float(np.matmul(intTrec_i.transpose(), link_flows).flatten()[0])

    return {
        'result': res,
        'D_r0': D_r0.flatten(),
        'D_r': D_r,
        'Xi': Xi,
        'link_flows': link_flows,
        'intTrec_i': intTrec_i,
        'objective': objective(D_r),
        'eq_max_abs': float(np.max(np.abs(eq_residual(D_r)))),
        'ineq_min': float(np.min(ineq_residual(D_r))),
        'total_od_demand': float(np.sum(D_od)),
        'total_route_demand': float(np.sum(D_r)),
        'total_travel_time': total_travel_time,
    }


In [ ]:
runs = {}
summary_rows = []

for label, config in OPTIMIZER_CONFIGS.items():
    print(f'Running {label} -> {config["method"]}')
    run = run_sue_optimizer(config['method'], config['options'], sue_inputs)
    runs[label] = run
    res = run['result']
    summary_rows.append({
        'label': label,
        'method': config['method'],
        'success': res.success,
        'status': res.status,
        'nit': getattr(res, 'nit', None),
        'nfev': getattr(res, 'nfev', None),
        'objective': run['objective'],
        'eq_max_abs': run['eq_max_abs'],
        'ineq_min': run['ineq_min'],
        'total_od_demand': run['total_od_demand'],
        'total_route_demand': run['total_route_demand'],
        'total_travel_time': run['total_travel_time'],
        'message': str(res.message),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('label').reset_index(drop=True)
display(summary_df)


In [ ]:
base_label = 'trust_constr_current'
compare_label = 'slsqp_loose_test'

comparison = pd.DataFrame({
    'route_id': np.arange(len(runs[base_label]['D_r'])),
    f'{base_label}_D_r': runs[base_label]['D_r'],
    f'{compare_label}_D_r': runs[compare_label]['D_r'],
})
comparison['abs_diff'] = np.abs(comparison[f'{base_label}_D_r'] - comparison[f'{compare_label}_D_r'])
comparison = comparison.sort_values('abs_diff', ascending=False)
display(comparison.head(20))

print('Interpretation help:')
print('- If SLSQP stops after 1 iteration and eq_max_abs is not clearly better, the loose tolerances are too permissive.')
print('- total_route_demand should be close to total_od_demand after OD constraints are enforced.')
print('- total_travel_time is the main cross-check for whether the optimizer result still gives plausible times.')
